# Task 2 — Evaluation of Effectiveness

## Task 2.1 — Our own implementation of the evaluation metrics

The brief requires every metric to be implemented **by us**, separately from RecBole, for a consistent
evaluation across all models and hybrids — and the beyond-accuracy metrics are not in RecBole at all.
This notebook implements them from scratch and applies them to every model from Task 1.1–1.2 and every
hybrid from Task 1.3–1.4, reading the exported top-10 lists in `runs/lists/` and `runs/hybrids/`.

**Accuracy** — Precision, Recall, Hit, MRR, NDCG, **MAP**.
**Beyond-accuracy** — intra-list diversity (ILD), novelty (self-information), catalogue coverage.

As a correctness check, our accuracy numbers are compared against RecBole's own test metrics (as in
§1.8 of Task 1); they must agree to within rounding for every deterministic model.

### How to run

Run top to bottom in the `Python (recsys)` kernel. This notebook only reads the artefacts produced by
`Task1.ipynb` (run that first), so it needs no GPU and no training. Tasks 2.2–2.4 also use
`scikit-learn` and `scipy`. Everything is written to `runs/tables/`.

In [3]:
# =============================================================================
# 2.1.0  Paths and data (ground truth, item popularity, item genres)
# =============================================================================
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["axes.unicode_minus"] = False

PROJECT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "RecBole_DSAIT4335").is_dir())
RUNS = PROJECT / "runs"
ITEM_FILE = PROJECT / "RecBole_DSAIT4335" / "dataset" / "ml-100k" / "ml-100k.item"
K = 10
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)

def ground_truth(split):
    f = pd.read_csv(RUNS / "splits" / f"ml-100k.{split}.tsv", sep="\t", dtype=str)
    gt = {}
    for u, i in zip(f.iloc[:, 0], f.iloc[:, 1]):
        gt.setdefault(u, set()).add(i)
    return gt

GT_TEST = ground_truth("test")

# item popularity for novelty: share of TRAIN interactions on each item, p(i) (lecture definition)
_train = pd.read_csv(RUNS / "splits" / "ml-100k.train.tsv", sep="\t", dtype=str)
_counts = _train.iloc[:, 1].value_counts()
POP_PROB = (_counts / int(_counts.sum())).to_dict()

# item genres for ILD: parse the `class` token_seq field of ml-100k.item
_it = pd.read_csv(ITEM_FILE, sep="\t", dtype=str)
_idc = [c for c in _it.columns if c.startswith("item_id")][0]
_clc = [c for c in _it.columns if c.startswith("class")][0]
GENRES = {str(i): frozenset(str(c).split()) if isinstance(c, str) and c else frozenset()
          for i, c in zip(_it[_idc], _it[_clc])}
CATALOG = set(GENRES)

print(f"catalog {len(CATALOG)} items | {len(GT_TEST)} test users | train interactions {int(_counts.sum())}")

catalog 1682 items | 943 test users | train interactions 80808


### Accuracy metrics

For a user with held-out relevant set $R$ and a top-$k$ list, with $hit_r=1$ if the item at rank $r$ is
relevant:

* **Precision@k** $= \frac{1}{k}\sum_r hit_r$, **Recall@k** $= \frac{1}{|R|}\sum_r hit_r$,
  **Hit@k** $= \mathbb{1}[\sum_r hit_r>0]$.
* **MRR@k** $= 1/\text{rank of first hit}$.
* **NDCG@k** $= DCG/IDCG$, $DCG=\sum_r hit_r/\log_2(r+1)$, $IDCG$ the same for a perfect ranking.
* **MAP@k** $= \frac{1}{\min(|R|,k)}\sum_r hit_r\cdot\text{Precision@}r$ — rewards relevant items ranked
  early, averaged over users.

The implementation is verified against RecBole's own reported numbers.

In [5]:
# =============================================================================
# 2.1.1  Accuracy metrics (incl. MAP) + verification against RecBole
# =============================================================================
def accuracy_metrics(lists, truth=GT_TEST, k=K):
    """Per-user binary-relevance accuracy metrics; returns (means dict, per-user DataFrame)."""
    disc = 1.0 / np.log2(np.arange(1, k + 1) + 1.0)
    rows = []
    for user, rec in lists.items():
        rel = truth.get(str(user))
        if not rel:
            continue
        top = list(rec["items"])[:k]
        hits = np.fromiter((1.0 if it in rel else 0.0 for it in top), float, len(top))
        nh = hits.sum()
        idcg = disc[:min(len(rel), k)].sum()
        first = np.flatnonzero(hits > 0)
        csum = np.cumsum(hits)
        ap = ((csum / np.arange(1, len(top) + 1)) * hits).sum() / min(len(rel), k)   # MAP
        rows.append({"user": str(user),
                     f"precision@{k}": nh / k, f"recall@{k}": nh / len(rel),
                     f"hit@{k}": 1.0 if nh else 0.0,
                     f"mrr@{k}": 1.0 / (first[0] + 1) if first.size else 0.0,
                     f"ndcg@{k}": float((hits * disc[:len(top)]).sum() / idcg) if idcg > 0 else 0.0,
                     f"map@{k}": float(ap)})
    pu = pd.DataFrame(rows)
    names = [f"{m}@{k}" for m in ("precision", "recall", "hit", "mrr", "ndcg", "map")]
    return {m: float(pu[m].mean()) for m in names}, pu

def load_list(path):
    return json.load(open(path))["recommendations"]

MODELS = {p.stem.replace(f"_top{K}", ""): load_list(p)
          for p in sorted((RUNS / "lists").glob(f"*_top{K}.json"))}

# verify vs RecBole's own test numbers (Random excluded: its ranking is not reproducible)
_baseline = json.load(open(RUNS / "results" / "baseline.json"))
_acc5 = [f"{m}@{K}" for m in ("recall", "mrr", "ndcg", "hit", "precision")]
_gap = 0.0
for _rec in _baseline:
    if _rec["run_id"] in MODELS and _rec["run_id"] != "Random":
        _ours, _ = accuracy_metrics(MODELS[_rec["run_id"]])
        _gap = max(_gap, max(abs(_ours[m] - float(_rec["recbole_test"][m])) for m in _acc5))
assert _gap < 5e-3, f"accuracy disagrees with RecBole (gap {_gap:.4f})"
print(f"accuracy verified against RecBole: max |ours - RecBole| = {_gap:.2e} (deterministic models)")

accuracy verified against RecBole: max |ours - RecBole| = 3.17e-03 (deterministic models)


### Beyond-accuracy metrics

These judge the *character* of a list, not its correctness, so they need side information rather than the
ground truth:

* **Intra-list diversity (ILD@k)** — mean pairwise distance between items in a list, using genre sets and
  Jaccard distance $d(i,j)=1-\frac{|G_i\cap G_j|}{|G_i\cup G_j|}$. Higher = more varied list.
* **Novelty@k (self-information / surprisal)** — mean of $-\log_2 p(i)$ over recommended items, where
  $p(i)$ is item $i$'s share of training interactions. Higher = the list surfaces rarer items.
* **Catalogue coverage@k** — fraction of the whole catalogue that appears in *any* user's list. A
  system-level diversity (aggregate diversity) measure.

In [7]:
# =============================================================================
# 2.1.2  Beyond-accuracy metrics (ILD genre-diversity, novelty, catalogue coverage)
# =============================================================================
def _jaccard_dist(a, b):
    u = len(a | b)
    return 1.0 - len(a & b) / u if u else 0.0

def beyond_accuracy_metrics(lists, k=K):
    ild, nov, shown = [], [], set()
    for rec in lists.values():
        top = list(rec["items"])[:k]
        shown.update(top)
        if len(top) > 1:
            g = [GENRES.get(it, frozenset()) for it in top]
            ild.append(float(np.mean([_jaccard_dist(g[a], g[b])
                                      for a in range(len(g)) for b in range(a + 1, len(g))])))
        si = [-np.log2(POP_PROB[it]) for it in top if POP_PROB.get(it, 0) > 0]
        if si:
            nov.append(float(np.mean(si)))
    return {f"ild@{k}": float(np.mean(ild)) if ild else 0.0,
            f"novelty@{k}": float(np.mean(nov)) if nov else 0.0,
            f"coverage@{k}": len(shown & CATALOG) / len(CATALOG)}

def all_metrics(lists, k=K):
    return {**accuracy_metrics(lists, k=k)[0], **beyond_accuracy_metrics(lists, k=k)}

In [8]:
# =============================================================================
# 2.1.3  Apply every metric to every model and hybrid -> one comparison table
# =============================================================================
HYBRIDS = {p.stem: load_list(p) for p in sorted((RUNS / "hybrids").glob("*.json"))
           if p.stem.startswith(("lists_", "task14_"))}

def _nice(name):
    return (name.replace("lists_", "§1.3 ").replace("task14_", "§1.4 ")
                .replace("_", " ").replace("ridgea10.0", "ridge").replace("logisticC1.0", "logistic"))

_rows = []
for rid, lst in MODELS.items():
    _rows.append({"run": rid, "group": "model", **all_metrics(lst)})
for name, lst in HYBRIDS.items():
    _rows.append({"run": _nice(name), "group": "hybrid", **all_metrics(lst)})

METRICS_ALL = pd.DataFrame(_rows).set_index("run").sort_values(f"ndcg@{K}", ascending=False)
METRICS_ALL.to_csv(RUNS / "tables" / "table_2_1_metrics.csv")
print("saved runs/tables/table_2_1_metrics.csv")
METRICS_ALL.round(4)

saved runs/tables/table_2_1_metrics.csv


In [9]:
# =============================================================================
# 2.1.4  Figure -- the accuracy vs beyond-accuracy trade-off
# =============================================================================
_d = METRICS_ALL.drop(index=[r for r in ("Random",) if r in METRICS_ALL.index])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for ax, (xcol, xlab) in zip(axes, [(f"coverage@{K}", "catalogue coverage"), (f"novelty@{K}", "novelty (self-information)")]):
    for grp, mk, col in [("model", "o", "#6b6b6b"), ("hybrid", "^", "#08519c")]:
        sub = _d[_d["group"] == grp]
        ax.scatter(sub[xcol], sub[f"ndcg@{K}"], marker=mk, c=col, s=55, label=grp, alpha=0.85)
    for name, r in _d.iterrows():
        ax.annotate(name, (r[xcol], r[f"ndcg@{K}"]), fontsize=7, xytext=(3, 3), textcoords="offset points")
    ax.set_xlabel(xlab); ax.set_ylabel(f"NDCG@{K}"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
axes[0].set_title("Accuracy vs coverage"); axes[1].set_title("Accuracy vs novelty")
fig.suptitle("Task 2.1 - accuracy and beyond-accuracy across models and hybrids")
fig.tight_layout()
fig.savefig(RUNS / "tables" / "fig_task21.png", dpi=130, bbox_inches="tight")
plt.show()
print("saved runs/tables/fig_task21.png")

cell_8:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
saved runs/tables/fig_task21.png


### Reading the Task 2.1 table — observations

* **Accuracy and beyond-accuracy pull in opposite directions.** The most accurate runs (EASE and the
  §1.3 weighted hybrid) have the *lowest* catalogue coverage and among the lowest novelty — they
  concentrate on a small set of popular, broadly-liked items. The graph/pairwise models (BPR, NGCF)
  are less accurate but cover 2–4× more of the catalogue and recommend rarer items.
* **The §1.4 hybrids sit where their mechanism predicts.** *Mixed* and *switching* trade a little NDCG
  for noticeably higher coverage and novelty than EASE — the diversity benefit that motivated them,
  now quantified. *Cascade* with a small shortlist stays close to EASE on every axis.
* **ILD (genre diversity) is high and flat (~0.80) for everything**, including Random: with only ~19
  genres any 10 movies are genre-varied, so ILD barely separates the systems on ML-100K. Coverage and
  novelty are the discriminating beyond-accuracy metrics here.
* **Caveat on the Random baseline:** RecBole's `Random` returns almost the same list to every user
  (only ~19 distinct items appear), so its catalogue coverage is *low*, not high — a quirk of the
  implementation to keep in mind when using it as a baseline in Task 2.2.

## Task 2.2 — Best models against the naïve baselines

Task 2.1 scored every exported list. Task 2.2 is the comparison the brief asks for: the **best-performing** system of each family, judged against `Random` and `Pop`.

**Who counts as best.** The project's selection metric is validation MRR@10 (Task 1.2 / 1.5). Test numbers are only reported, never used to pick a winner. Concretely:

* one individual model per family — the tuned configuration when Task 1.2 tuned it, otherwise the default (`EASE`, `Pop`, `Random`);
* one weighted hybrid — the fusion method with the highest **validation** MRR@10, not the highest sampled-negative AUC and not the highest test NDCG;
* one switching, cascade and mixed hybrid — the Task 1.5 validation winners (the lists saved for Task 2.1 are the *pre-tuning* defaults, so they are rebuilt here).

**How the comparison is read.** A mean gap is treated as reliable only when a user-level bootstrap 95% interval for the paired difference excludes zero. With 943 users a tiny gap can have a small p-value; the interval is the claim about magnitude. Genre ILD is reported in the csv and is not used to rank systems: Task 2.1 already showed it is flat on ML-100K.

In [12]:
# =============================================================================
# 2.2  Validation-selected systems, then one test comparison to Pop and Random
# =============================================================================
import warnings
from io import StringIO

import numpy as np
from scipy.optimize import nnls
from scipy.stats import wilcoxon
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold

warnings.filterwarnings("ignore", category=FutureWarning)

HYBRID_BASE = ["EASE", "NGCF", "ItemKNN-item", "BPR", "NeuMF"]
HYBRID_RUN_IDS = [f"tuned-{m}" for m in HYBRID_BASE]
SHORT = [m.replace("tuned-", "") for m in HYBRID_RUN_IDS]


def gt(split):
    frame = pd.read_csv(RUNS / "splits" / f"ml-100k.{split}.tsv", sep="\t", dtype=str)
    truth = {}
    for user, item in zip(frame.iloc[:, 0], frame.iloc[:, 1]):
        truth.setdefault(user, set()).add(item)
    return truth


GT_VALID = gt("valid")
_train = pd.read_csv(RUNS / "splits" / "ml-100k.train.tsv", sep="\t", dtype=str)
PROFILE = _train.groupby(_train.columns[0]).size().to_dict()
_head_n = max(1, int(round(0.1 * len(POP_PROB))))
HEAD = set(sorted(POP_PROB, key=POP_PROB.get, reverse=True)[:_head_n])


def load_score_matrices(run_ids):
    loaded = {}
    for run_id in run_ids:
        with np.load(RUNS / "scores" / f"{run_id}.npz") as npz:
            loaded[run_id] = (
                npz["scores_test"], npz["scores_valid"],
                np.asarray(npz["users_test"], dtype=str),
                np.asarray(npz["users_valid"], dtype=str),
                np.asarray(npz["items"], dtype=str),
            )
    first = loaded[run_ids[0]]
    for run_id, entry in loaded.items():
        assert np.array_equal(entry[2], first[2]), run_id
        assert np.array_equal(entry[3], first[3]), run_id
        assert np.array_equal(entry[4], first[4]), run_id
    return first[2], first[3], first[4], {r: (e[0], e[1]) for r, e in loaded.items()}


def per_user_minmax(scores):
    masked = np.where(np.isfinite(scores), scores, np.nan).astype(np.float32)
    with np.errstate(all="ignore"):
        low = np.nanmin(masked, axis=1, keepdims=True)
        high = np.nanmax(masked, axis=1, keepdims=True)
    span = np.where(high > low, high - low, np.float32(1.0))
    return (masked - low) / span


def topk_from_scores(fused, users, k=K):
    top = np.argsort(-fused, axis=1, kind="stable")[:, :k]
    return {
        str(user): {"items": [str(ITEMS[j]) for j in row],
                    "scores": [float(fused[i, j]) for j in row]}
        for i, (user, row) in enumerate(zip(users, top))
    }


def fuse_scores(run_ids, weights, normalized, mask):
    fused = np.zeros(mask.shape, dtype=np.float64)
    for run_id, weight in zip(run_ids, weights):
        fused += np.nan_to_num(weight * normalized[run_id], nan=0.0)
    return np.where(mask, fused, -np.inf)


USERS_TEST, USERS_VALID, ITEMS, SCORES = load_score_matrices(
    HYBRID_RUN_IDS + ["tuned-SLIMElastic"])
CAND_TEST = np.isfinite(SCORES["tuned-EASE"][0])
CAND_VALID = np.isfinite(SCORES["tuned-EASE"][1])
NORM_TEST = {r: per_user_minmax(S[0]) for r, S in SCORES.items()}
NORM_VALID = {r: per_user_minmax(S[1]) for r, S in SCORES.items()}
ITEM_COLUMN = {token: j for j, token in enumerate(ITEMS)}
TEST_ROW = {user: i for i, user in enumerate(USERS_TEST)}
VALID_ROW = {user: i for i, user in enumerate(USERS_VALID)}

for _run in SCORES:
    assert np.array_equal(np.isfinite(SCORES[_run][0]), CAND_TEST), _run
    assert np.array_equal(np.isfinite(SCORES[_run][1]), CAND_VALID), _run

# Our metric on the score matrix must reproduce RecBole's validation MRR (4 d.p. in Table 1.2).
_mrr_check = {}
for _run, _published in (("tuned-EASE", 0.4201), ("tuned-SLIMElastic", 0.4216)):
    _lists = topk_from_scores(np.where(CAND_VALID, NORM_VALID[_run], -np.inf), USERS_VALID)
    _mrr = accuracy_metrics(_lists, GT_VALID)[0]["mrr@10"]
    assert abs(_mrr - _published) < 5e-4, (_run, _mrr, _published)
    _mrr_check[_run] = _mrr
print("validation MRR from scores:", {k: round(v, 4) for k, v in _mrr_check.items()},
      "(Table 1.2: EASE 0.4201, SLIMElastic 0.4216)")


def build_regression_data(run_ids, n_neg=4, seed=2020):
    """Validation positives + 4 uniform negatives. Same construction as Task 1.3."""
    rng = np.random.default_rng(seed)
    features, labels, groups = [], [], []
    n_neg_rows = n_contaminated = 0
    for user in USERS_VALID:
        if user not in GT_VALID or user not in TEST_ROW:
            continue
        positives = [ITEM_COLUMN[i] for i in GT_VALID[user] if i in ITEM_COLUMN]
        if not positives:
            continue
        neg_pool = np.setdiff1d(np.flatnonzero(CAND_TEST[TEST_ROW[user]]), positives)
        if neg_pool.size == 0:
            continue
        take = rng.choice(neg_pool, size=min(n_neg * len(positives), neg_pool.size), replace=False)
        positive_set = set(positives)
        test_items = GT_TEST.get(user, ())
        for column in [*positives, *take]:
            features.append([NORM_VALID[r][VALID_ROW[user], column] for r in run_ids])
            label = 1.0 if column in positive_set else 0.0
            labels.append(label)
            groups.append(user)
            if label == 0.0:
                n_neg_rows += 1
                n_contaminated += int(ITEMS[column] in test_items)
    X = np.asarray(features, dtype=np.float64)
    y = np.asarray(labels, dtype=np.float64)
    assert np.isfinite(X).all()
    return X, y, np.asarray(groups), n_contaminated, n_neg_rows


def _specs():
    specs = {f"ridge(a={a})": Ridge(alpha=a) for a in (0.01, 0.1, 1.0, 10.0)}
    specs.update({f"logistic(C={c})": LogisticRegression(C=c, max_iter=2000)
                  for c in (0.1, 1.0, 10.0)})
    specs["nnls"] = None
    return specs


def _fit_coef(name, X, y):
    estimator = _specs()[name]
    if estimator is None:
        w, _ = nnls(X, y)
        return np.asarray(w, dtype=float)
    return np.asarray(clone(estimator).fit(X, y).coef_, dtype=float).ravel()


X_TRAIN, Y_TRAIN, GROUPS, N_CONTAMINATED, N_NEG_ROWS = build_regression_data(HYBRID_RUN_IDS)
print(f"regression sample: {X_TRAIN.shape[0]} rows, {int(Y_TRAIN.sum())} positives, "
      f"{len(np.unique(GROUPS))} users")
print(f"negatives that are actually test items: {N_CONTAMINATED}/{N_NEG_ROWS} "
      f"({N_CONTAMINATED / N_NEG_ROWS:.1%}) — unknown at validation time, so they stay")

folds = list(GroupKFold(n_splits=5).split(X_TRAIN, Y_TRAIN, GROUPS))
cv_auc, coefficients, fold_coefs = {}, {}, {}
for name in _specs():
    aucs, per_fold = [], []
    for train_idx, test_idx in folds:
        w = _fit_coef(name, X_TRAIN[train_idx], Y_TRAIN[train_idx])
        per_fold.append(w)
        aucs.append(roc_auc_score(Y_TRAIN[test_idx], X_TRAIN[test_idx] @ w))
    cv_auc[name] = float(np.mean(aucs))
    coefficients[name] = _fit_coef(name, X_TRAIN, Y_TRAIN)
    fold_coefs[name] = np.vstack(per_fold)

COEF = pd.DataFrame(coefficients, index=HYBRID_RUN_IDS).T
_published = pd.read_csv(StringIO((RUNS / "hybrids" / "weights.csv").read_text().split("# relative")[0]),
                         comment="#").rename(columns={"Unnamed: 0": "method"}).set_index("method")
_gap = float((COEF - _published.loc[COEF.index, HYBRID_RUN_IDS]).abs().to_numpy().max())
assert _gap < 1e-8, _gap
print(f"refit coefficients match runs/hybrids/weights.csv (max abs gap {_gap:.1e})")


def ranking_metrics(weights, run_ids, split):
    if split == "test":
        norm, cand, users, truth = NORM_TEST, CAND_TEST, USERS_TEST, GT_TEST
    else:
        norm, cand, users, truth = NORM_VALID, CAND_VALID, USERS_VALID, GT_VALID
    lists = topk_from_scores(fuse_scores(run_ids, weights, norm, cand), users)
    summary, per_user = accuracy_metrics(lists, truth)
    return summary, per_user, lists


fusion_rows, FUSION_LISTS_TEST = [], {}
_candidates = {"equal": np.full(len(HYBRID_RUN_IDS), 1.0 / len(HYBRID_RUN_IDS))}
_candidates.update({name: COEF.loc[name].to_numpy() for name in COEF.index})
for name, w in _candidates.items():
    valid, _, _ = ranking_metrics(w, HYBRID_RUN_IDS, "valid")
    test, _, lists_t = ranking_metrics(w, HYBRID_RUN_IDS, "test")
    fusion_rows.append({
        "method": name, "cv_auc": cv_auc.get(name, np.nan),
        "valid_mrr@10": valid["mrr@10"], "valid_ndcg@10": valid["ndcg@10"],
        "test_mrr@10": test["mrr@10"], "test_ndcg@10": test["ndcg@10"],
        "test_recall@10": test["recall@10"],
    })
    FUSION_LISTS_TEST[name] = lists_t
FUSION = pd.DataFrame(fusion_rows).sort_values(
    ["valid_mrr@10", "cv_auc"], ascending=False).reset_index(drop=True)
PRIMARY = FUSION.loc[0, "method"]
primary_w = _candidates[PRIMARY]
print(f"\nweighted hybrid selected by validation MRR@10: {PRIMARY}")
print("AUC would have picked the same logistic model; test NDCG would have picked "
      f"{FUSION.sort_values('test_ndcg@10', ascending=False).iloc[0]['method']}, which is not used.")
print(FUSION.round(4).to_string(index=False))


validation MRR from scores: {'tuned-EASE': 0.4201, 'tuned-SLIMElastic': 0.4216} (Table 1.2: EASE 0.4201, SLIMElastic 0.4216)
regression sample: 47980 rows, 9596 positives, 943 users
negatives that are actually test items: 548/38384 (1.4%) — unknown at validation time, so they stay
refit coefficients match runs/hybrids/weights.csv (max abs gap 2.5e-12)

weighted hybrid selected by validation MRR@10: logistic(C=1.0)
AUC would have picked the same logistic model; test NDCG would have picked ridge(a=10.0), which is not used.
          method  cv_auc  valid_mrr@10  valid_ndcg@10  test_mrr@10  test_ndcg@10  test_recall@10
 logistic(C=1.0)  0.9205        0.4211         0.2599       0.5295        0.3306          0.2779
logistic(C=10.0)  0.9205        0.4209         0.2598       0.5300        0.3307          0.2775
 logistic(C=0.1)  0.9204        0.4208         0.2599       0.5286        0.3310          0.2780
   ridge(a=10.0)  0.9187        0.4185         0.2594       0.5357        0.3338     

In [13]:
# =============================================================================
# 2.2b  Test metrics, paired gaps, figure
# =============================================================================
def hybrid_switching(threshold, low_model, high_model, norm, cand, users):
    """Profile size above the threshold -> high_model, otherwise low_model. Same rule as Task 1.4."""
    lo, hi = norm[f"tuned-{low_model}"], norm[f"tuned-{high_model}"]
    pick_hi = np.array([PROFILE.get(str(u), 0) > threshold for u in users])[:, None]
    fused = np.where(pick_hi, hi, lo)
    fused = np.where(cand & np.isfinite(fused), fused, -np.inf)
    return topk_from_scores(fused, users)


def hybrid_cascade(stage1, stage2, n_candidates, norm, cand, users):
    s1 = np.where(cand, np.nan_to_num(norm[f"tuned-{stage1}"], nan=-np.inf), -np.inf)
    shortlist = np.argsort(-s1, axis=1, kind="stable")[:, :n_candidates]
    s2 = np.where(cand, np.nan_to_num(norm[f"tuned-{stage2}"], nan=-np.inf), -np.inf)
    fused = np.full(s1.shape, -np.inf)
    rows = np.arange(s1.shape[0])[:, None]
    fused[rows, shortlist] = s2[rows, shortlist]
    return topk_from_scores(fused, users)


def hybrid_mixed(models, norm, cand, users, k=K):
    rank = {
        m: np.argsort(-np.where(cand, np.nan_to_num(norm[f"tuned-{m}"], nan=-np.inf), -np.inf),
                      axis=1, kind="stable")
        for m in models
    }
    out = {}
    for i, u in enumerate(users):
        picked, seen, ptr = [], set(), {m: 0 for m in models}
        while len(picked) < k:
            progressed = False
            for m in models:
                order = rank[m][i]
                while ptr[m] < order.shape[0]:
                    col = int(order[ptr[m]])
                    ptr[m] += 1
                    if cand[i, col] and col not in seen:
                        picked.append(col)
                        seen.add(col)
                        progressed = True
                        break
                if len(picked) >= k:
                    break
            if not progressed:
                break
        out[str(u)] = {"items": [str(ITEMS[c]) for c in picked],
                       "scores": [float(k - j) for j in range(len(picked))]}
    return out


_V = dict(norm=NORM_VALID, cand=CAND_VALID, users=USERS_VALID)
_T = dict(norm=NORM_TEST, cand=CAND_TEST, users=USERS_TEST)
_prof = np.array(list(PROFILE.values()))
TAU_GRID = sorted({int(np.quantile(_prof, q)) for q in (0.2, 0.35, 0.5, 0.65, 0.8)})
N_GRID = [10, 20, 50, 100, 200, 500]
_other = []
for _lo, _hi in [("EASE", "NGCF"), ("ItemKNN-item", "NGCF")]:
    for _tau in TAU_GRID:
        _vm = accuracy_metrics(hybrid_switching(_tau, _lo, _hi, **_V), GT_VALID)[0]["mrr@10"]
        _other.append(("switching", f"switch {_lo}/{_hi} tau={_tau}", _vm,
                       lambda lo=_lo, hi=_hi, tau=_tau: hybrid_switching(tau, lo, hi, **_T)))
for _s1, _s2 in [("EASE", "NGCF"), ("EASE", "BPR")]:
    for _n in N_GRID:
        _vm = accuracy_metrics(hybrid_cascade(_s1, _s2, _n, **_V), GT_VALID)[0]["mrr@10"]
        _other.append(("cascade", f"cascade {_s1}->{_s2} N={_n}", _vm,
                       lambda s1=_s1, s2=_s2, n=_n: hybrid_cascade(s1, s2, n, **_T)))
for _partner in [m for m in HYBRID_BASE if m != "EASE"]:
    _vm = accuracy_metrics(hybrid_mixed(["EASE", _partner], **_V), GT_VALID)[0]["mrr@10"]
    _other.append(("mixed", f"mixed EASE+{_partner}", _vm,
                   lambda partner=_partner: hybrid_mixed(["EASE", partner], **_T)))

_pub15 = pd.read_csv(RUNS / "tables" / "table_1_5_tuning.csv")
_ours15 = pd.DataFrame([{"family": f, "config": c, "valid_mrr@10": m} for f, c, m, _ in _other])
_merged = _ours15.merge(_pub15, on=["family", "config"], suffixes=("_ours", "_pub"))
_gap15 = float((_merged["valid_mrr@10_ours"] - _merged["valid_mrr@10_pub"]).abs().max())
assert _gap15 < 1e-8, _gap15
print(f"Task 1.5 validation grid reproduced (max abs gap {_gap15:.1e})")

# best individual per family, by the validation MRR already stored in Table 1.2
_t12 = pd.read_csv(RUNS / "tables" / "table_1_2_tuning.csv")
_ind = []
for _, r in _t12.iterrows():
    tuned = r["tuned"] == True or r["tuned"] == "True"
    _ind.append({
        "run": f"tuned-{r['run_id']}" if tuned else r["run_id"],
        "family": r["family"],
        "valid_mrr@10": float(r["tuned_valid_mrr"] if tuned else r["default_valid_mrr"]),
    })
IND = pd.DataFrame(_ind)
BEST_IND = (IND[IND["family"] != "Naive baseline"]
            .sort_values("valid_mrr@10", ascending=False)
            .groupby("family", as_index=False).head(1))
BASELINES = IND[IND["family"] == "Naive baseline"]

SELECTED = {}
for _, r in pd.concat([BASELINES, BEST_IND]).iterrows():
    SELECTED[r["run"]] = load_list(RUNS / "lists" / f"{r['run']}_top{K}.json")
SELECTED[f"weighted:{PRIMARY}"] = FUSION_LISTS_TEST[PRIMARY]
WINNERS = {}
for fam, config, mrr, builder in _other:
    if fam not in WINNERS or mrr > WINNERS[fam][0]:
        WINNERS[fam] = (mrr, config, builder)
for fam, (mrr, config, builder) in WINNERS.items():
    SELECTED[f"{fam}:{config}"] = builder()

LABELS = {
    "Random": "Random", "Pop": "Popularity", "EASE": "EASE",
    "tuned-SLIMElastic": "SLIMElastic", "tuned-ItemKNN-user": "ItemKNN-user",
    "tuned-NGCF": "NGCF", "tuned-NeuMF": "NeuMF", "tuned-BPR": "BPR", "tuned-FISM": "FISM",
    f"weighted:{PRIMARY}": "Weighted logistic",
    f"switching:{WINNERS['switching'][1]}": "Switch EASE/NGCF",
    f"cascade:{WINNERS['cascade'][1]}": "Cascade EASE-NGCF",
    f"mixed:{WINNERS['mixed'][1]}": "Mixed EASE+NGCF",
}
assert set(SELECTED) == set(LABELS), set(SELECTED) ^ set(LABELS)
POP_LISTS = {u: set(rec["items"]) for u, rec in SELECTED["Pop"].items()}


def list_extras(lists):
    head, jac = [], []
    for user, rec in lists.items():
        top = list(rec["items"])[:K]
        head.append(float(np.mean([it in HEAD for it in top])))
        both = set(top) | POP_LISTS[str(user)]
        jac.append(len(set(top) & POP_LISTS[str(user)]) / len(both))
    return {f"head_frac@{K}": float(np.mean(head)), f"jaccard_pop@{K}": float(np.mean(jac))}


rows, PER_USER = [], {}
for name, lists in SELECTED.items():
    acc, pu = accuracy_metrics(lists, GT_TEST)
    metrics = {**acc, **beyond_accuracy_metrics(lists)}
    pu = pu.set_index("user").sort_index()
    PER_USER[name] = pu
    if name in set(IND["run"]):
        valid, kind = float(IND.set_index("run").loc[name, "valid_mrr@10"]), (
            "baseline" if name in ("Pop", "Random") else "individual")
    elif name.startswith("weighted"):
        valid, kind = float(FUSION.loc[0, "valid_mrr@10"]), "hybrid"
    else:
        valid, kind = float(WINNERS[name.split(":")[0]][0]), "hybrid"
    rows.append({"system": name, "label": LABELS[name], "kind": kind,
                 "valid_mrr@10": valid, **metrics, **list_extras(lists)})

TABLE_22 = pd.DataFrame(rows).sort_values("valid_mrr@10", ascending=False)
TABLE_22.to_csv(RUNS / "tables" / "table_2_2_selected.csv", index=False)

# analytical floor of a uniform per-user ranking (RecBole's Random list is one degenerate draw)
_rel = np.array([len(GT_TEST[u]) for u in USERS_TEST], dtype=float)
_cands = CAND_TEST.sum(1).astype(float)
_exp_recall = float(np.mean(K / _cands))
_exp_precision = float(np.mean(_rel / _cands))
print(f"uniform ranking, expected recall@{K} = {_exp_recall:.4f}, "
      f"expected precision@{K} ≈ {_exp_precision:.4f}")
print(f"RecBole Random list: recall {TABLE_22.set_index('system').loc['Random', 'recall@10']:.4f}, "
      f"coverage {TABLE_22.set_index('system').loc['Random', 'coverage@10']:.4f}")


def paired(a, b, n_boot=2000, seed=2020):
    joined = pd.concat([a.rename("a"), b.rename("b")], axis=1).dropna()
    diff = (joined["a"] - joined["b"]).to_numpy()
    rng = np.random.default_rng(seed)
    draws = diff[rng.integers(0, len(diff), size=(n_boot, len(diff)))].mean(axis=1)
    lo, hi = np.quantile(draws, [0.025, 0.975])
    nonzero = diff[diff != 0]
    p = 1.0 if nonzero.size == 0 else float(wilcoxon(nonzero, alternative="two-sided").pvalue)
    return {"mean_diff": float(diff.mean()), "ci_lo": float(lo), "ci_hi": float(hi), "p": p,
            "frac_better": float((diff > 0).mean()), "frac_tie": float((diff == 0).mean())}


BEST_INDIVIDUAL = BEST_IND.sort_values("valid_mrr@10", ascending=False).iloc[0]["run"]
_focus = [
    ("Pop", "ndcg@10"), ("Pop", "recall@10"),
    (BEST_INDIVIDUAL, "ndcg@10"), ("EASE", "ndcg@10"), ("EASE", "mrr@10"), ("EASE", "recall@10"),
]
comp_rows = []
for name in TABLE_22["system"]:
    if name == "Pop":
        continue
    for versus, metric in _focus:
        if name == versus:
            continue
        comp_rows.append({"system": LABELS[name], "versus": LABELS.get(versus, versus),
                          "metric": metric, **paired(PER_USER[name][metric], PER_USER[versus][metric])})
COMP_22 = pd.DataFrame(comp_rows)
COMP_22.to_csv(RUNS / "tables" / "table_2_2_paired.csv", index=False)
print(f"best individual by validation MRR: {LABELS[BEST_INDIVIDUAL]}")
print("\npaired test gaps for the top cluster (mean difference and 95% bootstrap interval)")
_top = {"SLIMElastic", "EASE", "Weighted logistic", "Switch EASE/NGCF", "Cascade EASE-NGCF", "Mixed EASE+NGCF"}
print(COMP_22[COMP_22["system"].isin(_top) & COMP_22["metric"].isin(["ndcg@10", "mrr@10"])]
      .round(4).to_string(index=False))

_show = ["label", "kind", "valid_mrr@10", "ndcg@10", "recall@10", "mrr@10", "hit@10",
         "novelty@10", "coverage@10", "head_frac@10", "jaccard_pop@10"]
print("\nselected systems, ordered by validation MRR@10")
print(TABLE_22[_show].round(4).to_string(index=False))

# figure: same system order on accuracy and on catalogue coverage
_plot = TABLE_22.sort_values("ndcg@10", ascending=True)
_colors = {"baseline": "#bdbdbd", "individual": "#6b6b6b", "hybrid": "#08519c"}
_bar_colors = [_colors[k] for k in _plot["kind"]]
fig, axes = plt.subplots(1, 2, figsize=(12.2, 5.2), sharey=True)
axes[0].barh(_plot["label"], _plot["ndcg@10"], color=_bar_colors)
axes[0].axvline(float(TABLE_22.set_index("label").loc["Popularity", "ndcg@10"]),
                color="#e6550d", ls="--", lw=1)
axes[0].set_xlabel("test NDCG@10")
axes[0].set_title("Accuracy (dashed = Popularity)")
axes[1].barh(_plot["label"], _plot["coverage@10"], color=_bar_colors)
axes[1].set_xlabel("catalogue coverage@10")
axes[1].set_title("Coverage, same order")
for ax in axes:
    ax.grid(alpha=0.3, axis="x")
fig.suptitle("Task 2.2 - validation-selected systems against the naive baselines")
fig.tight_layout()
fig.savefig(RUNS / "tables" / "fig_task22.png", dpi=130, bbox_inches="tight")
plt.show()
print("saved runs/tables/table_2_2_selected.csv, table_2_2_paired.csv, fig_task22.png")


Task 1.5 validation grid reproduced (max abs gap 5.6e-17)
uniform ranking, expected recall@10 = 0.0063, expected precision@10 ≈ 0.0069
RecBole Random list: recall 0.0031, coverage 0.0113
best individual by validation MRR: SLIMElastic

paired test gaps for the top cluster (mean difference and 95% bootstrap interval)
           system      versus  metric  mean_diff   ci_lo   ci_hi      p  frac_better  frac_tie
      SLIMElastic  Popularity ndcg@10     0.2230  0.2090  0.2374 0.0000       0.7487    0.1760
      SLIMElastic        EASE ndcg@10    -0.0060 -0.0123  0.0003 0.0477       0.3510    0.2418
      SLIMElastic        EASE  mrr@10    -0.0130 -0.0281  0.0023 0.0553       0.1909    0.5822
Weighted logistic  Popularity ndcg@10     0.2301  0.2157  0.2443 0.0000       0.7561    0.1633
Weighted logistic SLIMElastic ndcg@10     0.0071 -0.0004  0.0146 0.0387       0.4305    0.2004
Weighted logistic        EASE ndcg@10     0.0011 -0.0050  0.0069 0.5297       0.3796    0.2418
Weighted logistic 

### Reading Task 2.2

Validation MRR puts tuned SLIMElastic (0.422), the logistic weighted hybrid (0.421) and EASE (0.420) in one cluster. Pairwise bootstrap intervals for their test-NDCG gaps all include zero. Ridge and equal weights look better on test NDCG and are not selected, because that comparison uses the test set.

Each beats Popularity by about 0.23 NDCG, and the interval excludes zero. Popularity is about 0.10 NDCG above RecBole's Random model. The gain is personalisation inside the head: the best lists take about 85% of their top-10 from the 10% most popular training items, but Jaccard with Popularity is only 0.13.

Other families beat Popularity by less and lose a reliable 0.04–0.08 NDCG to the linear item–item models. User-kNN and FISM are more concentrated on the head and cover less of the catalogue; NGCF, NeuMF and BPR cover more and are more novel. Switching, mixed and cascade do not beat the linear models. Cascade at N=10 keeps EASE's items and only lets NGCF reorder them, dropping MRR from 0.528 to 0.506. ILD is flat, including for Random. Random's novelty and 1% coverage come from a near-constant list of rare items; a uniform ranking would have recall near 0.006, not that list's 0.003.


## Task 2.3 — Coefficient analysis

The question is how each base model contributes to the **selected** weighted hybrid from Task 2.2. Three quantities answer different parts of it, and they are not interchangeable:

* **Weight.** The regression coefficient on the per-user min–max score. Reported with the standard deviation across the five user-grouped folds, because a large weight that flips between folds is not an identified contribution.
* **Score share.** The weight times that model's normalised score, averaged over the hybrid's test top-10. This is the share of the fused score on the items that are actually recommended.
* **Accuracy contribution.** Leave-one-out: refit without the model and measure the change in test NDCG. Shapley values split the same metric across all 2^5 subsets, with the empty hybrid scored as 0, so they also credit accuracy a model has on its own.

Sampled-negative AUC is not used to interpret the coefficients. NGCF's agreement with BPR (Task 1.3) is carried in as a variance-inflation factor: a coefficient on a duplicated signal is not a separate mechanism. The ridge fit is shown only as a contrast. It was not the validation-selected hybrid.


In [16]:
# =============================================================================
# 2.3  Coefficient analysis of the validation-selected weighted hybrid
# Three different questions, kept separate on purpose:
#   (1) weight      — the coefficient in the fused score
#   (2) score share — that weight times the model's score, on the items actually recommended
#   (3) accuracy    — leave-one-out and Shapley on the ranking metric
# =============================================================================
from math import factorial

from sklearn.linear_model import LinearRegression

assert PRIMARY.startswith("logistic"), PRIMARY
_C = float(PRIMARY.split("C=")[1].rstrip(")"))


def _logistic(X, y):
    return np.asarray(LogisticRegression(C=_C, max_iter=2000).fit(X, y).coef_, dtype=float).ravel()


fold = pd.DataFrame(fold_coefs[PRIMARY], columns=SHORT)
weight_share = primary_w / primary_w.sum()
COEF_SUMMARY = pd.DataFrame({
    "weight": primary_w,
    "fold_mean": fold.mean().to_numpy(),
    "fold_std": fold.std(ddof=1).to_numpy(),
    "weight_share": weight_share,
}, index=SHORT)

vif_rows = []
for i, name in enumerate(SHORT):
    y = X_TRAIN[:, i]
    others = np.delete(X_TRAIN, i, axis=1)
    r2 = float(LinearRegression().fit(others, y).score(others, y))
    pos, neg = Y_TRAIN == 1, Y_TRAIN == 0
    gap = float(X_TRAIN[pos, i].mean() - X_TRAIN[neg, i].mean())
    vif_rows.append({
        "model": name, "vif": 1.0 / (1.0 - r2), "r2_on_others": r2,
        "mean_score_pos": float(X_TRAIN[pos, i].mean()),
        "mean_score_neg": float(X_TRAIN[neg, i].mean()),
        "score_gap": gap, "gap_times_weight": gap * primary_w[i],
    })
VIF = pd.DataFrame(vif_rows).set_index("model")
COEF_SUMMARY = COEF_SUMMARY.join(VIF)

# contribution to the fused score of the items that make the test top-10
fused_test = fuse_scores(HYBRID_RUN_IDS, primary_w, NORM_TEST, CAND_TEST)
top_cols = np.argsort(-fused_test, axis=1, kind="stable")[:, :K]
contrib = np.zeros((len(USERS_TEST), K, len(HYBRID_RUN_IDS)))
for j, rid in enumerate(HYBRID_RUN_IDS):
    scores = np.nan_to_num(NORM_TEST[rid], nan=0.0)
    contrib[:, :, j] = primary_w[j] * scores[np.arange(len(USERS_TEST))[:, None], top_cols]
share = contrib / contrib.sum(axis=2, keepdims=True)
COEF_SUMMARY["top10_score_share"] = share.mean(axis=(0, 1))
COEF_SUMMARY["rank1_score_share"] = share[:, 0, :].mean(axis=0)

# leave-one-out: zero the weight, and refit the logistic model without that column
full_test = ranking_metrics(primary_w, HYBRID_RUN_IDS, "test")[0]
full_valid = ranking_metrics(primary_w, HYBRID_RUN_IDS, "valid")[0]
loo_rows = []
for j, name in enumerate(SHORT):
    w0 = primary_w.copy()
    w0[j] = 0.0
    zero_ndcg = ranking_metrics(w0, HYBRID_RUN_IDS, "test")[0]["ndcg@10"]
    keep = [i for i in range(len(SHORT)) if i != j]
    w_refit = _logistic(X_TRAIN[:, keep], Y_TRAIN)
    sub_ids = [HYBRID_RUN_IDS[i] for i in keep]
    refit_test = ranking_metrics(w_refit, sub_ids, "test")[0]["ndcg@10"]
    refit_valid = ranking_metrics(w_refit, sub_ids, "valid")[0]["mrr@10"]
    loo_rows.append({
        "model": name,
        "ndcg_change_if_zeroed": zero_ndcg - full_test["ndcg@10"],
        "ndcg_change_if_refit": refit_test - full_test["ndcg@10"],
        "valid_mrr_change_if_refit": refit_valid - full_valid["mrr@10"],
    })
LOO = pd.DataFrame(loo_rows).set_index("model")

# Shapley: fair split of the hybrid's metric across the five models.
# v(empty) = 0, so the values sum to the full-model metric and a model is credited
# for accuracy it can produce on its own, not only for what it adds on top of EASE.
def _subset_value(mask, cache):
    if mask in cache:
        return cache[mask]
    idx = [i for i in range(len(SHORT)) if mask & (1 << i)]
    if not idx:
        cache[mask] = (0.0, 0.0)
        return cache[mask]
    w = _logistic(X_TRAIN[:, idx], Y_TRAIN)
    sub_ids = [HYBRID_RUN_IDS[i] for i in idx]
    cache[mask] = (
        ranking_metrics(w, sub_ids, "test")[0]["ndcg@10"],
        ranking_metrics(w, sub_ids, "valid")[0]["mrr@10"],
    )
    return cache[mask]


_cache, shap = {}, np.zeros((len(SHORT), 2))
n_models = len(SHORT)
for mask in range(1 << n_models):
    base_ndcg, base_mrr = _subset_value(mask, _cache)
    for j in range(n_models):
        if mask & (1 << j):
            continue
        # |S|! (n-|S|-1)! / n!  — only defined when j is not already in S
        size = bin(mask).count("1")
        weight = factorial(size) * factorial(n_models - size - 1) / factorial(n_models)
        nxt_ndcg, nxt_mrr = _subset_value(mask | (1 << j), _cache)
        shap[j, 0] += weight * (nxt_ndcg - base_ndcg)
        shap[j, 1] += weight * (nxt_mrr - base_mrr)

SHAP = pd.DataFrame(shap, index=SHORT, columns=["shapley_test_ndcg", "shapley_valid_mrr"])
SHAP["share_of_test_ndcg"] = SHAP["shapley_test_ndcg"] / SHAP["shapley_test_ndcg"].sum()
assert abs(SHAP["shapley_test_ndcg"].sum() - full_test["ndcg@10"]) < 1e-9

CONTRIB = COEF_SUMMARY.join(LOO).join(SHAP)
# ridge is not the selected hybrid; it is the estimator whose coefficients tell a different story
CONTRIB["ridge_a10_weight"] = COEF.loc["ridge(a=10.0)"].to_numpy()
CONTRIB.to_csv(RUNS / "tables" / "table_2_3_contribution.csv")
print(f"selected fusion: {PRIMARY}")
print(CONTRIB.round(4).to_string())
print(f"\nsum of (score gap × weight) = {CONTRIB['gap_times_weight'].sum():.3f} "
      "on the validation regression sample")
print(f"Shapley test NDCG sums to {SHAP['shapley_test_ndcg'].sum():.4f} "
      f"(the hybrid's test NDCG is {full_test['ndcg@10']:.4f})")

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
x = np.arange(len(SHORT))
ridge_share = CONTRIB["ridge_a10_weight"] / CONTRIB["ridge_a10_weight"].abs().sum()
axes[0].bar(x - 0.18, CONTRIB["weight_share"], 0.36, label="logistic (selected)", color="#08519c")
axes[0].bar(x + 0.18, ridge_share, 0.36, label="ridge α=10 (not selected)", color="#9ecae1")
axes[0].axhline(0, color="k", lw=0.6)
axes[0].set_xticks(x)
axes[0].set_xticklabels(SHORT, rotation=15)
axes[0].set_ylabel("signed weight  /  sum |w|")
axes[0].set_title("Coefficients")
axes[0].legend(fontsize=8)
colors = ["#e6550d" if v < 0 else "#08519c" for v in CONTRIB["ndcg_change_if_refit"]]
axes[1].barh(SHORT, CONTRIB["ndcg_change_if_refit"], color=colors)
axes[1].axvline(0, color="k", lw=0.6)
axes[1].set_xlabel("change in test NDCG@10 if the model is removed and the rest refit")
axes[1].set_title("Incremental accuracy")
for ax in axes:
    ax.grid(alpha=0.3)
fig.suptitle(f"Task 2.3 - {PRIMARY}: weight is not the same thing as contribution")
fig.tight_layout()
fig.savefig(RUNS / "tables" / "fig_task23.png", dpi=130, bbox_inches="tight")
plt.show()
print("saved runs/tables/table_2_3_contribution.csv, fig_task23.png")


selected fusion: logistic(C=1.0)
              weight  fold_mean  fold_std  weight_share      vif  r2_on_others  mean_score_pos  mean_score_neg  score_gap  gap_times_weight  top10_score_share  rank1_score_share  ndcg_change_if_zeroed  ndcg_change_if_refit  valid_mrr_change_if_refit  shapley_test_ndcg  shapley_valid_mrr  share_of_test_ndcg  ridge_a10_weight
EASE          2.1315     2.1199    0.0680        0.2415   3.8992        0.7435          0.4961          0.1990     0.2971            0.6333             0.2089             0.2406                -0.0301               -0.0231                    -0.0293             0.0932             0.1098              0.2818            0.6171
NGCF          2.0889     2.0816    0.1395        0.2366  12.8678        0.9223          0.7203          0.3767     0.3436            0.7178             0.2407             0.2363                 0.0039                0.0027                    -0.0041             0.0594             0.0800              0.1797        

### Reading Task 2.3

The selected fusion is logistic regression with C=1. Fold-to-fold the coefficients barely move: EASE, NGCF, BPR and NeuMF each get about a quarter of the weight, and item-kNN about 2%. Equal weights are not an equal contribution to the ranking.

On the regression sample NeuMF separates validation positives from sampled negatives the most (score gap 0.56), so the logistic loss gives it the largest weight. EASE's scores are less separated on random negatives (gap 0.30) even though EASE orders the head better. Leave-one-out refits show the difference: removing EASE costs 0.023 test NDCG, and removing any other model moves NDCG by at most 0.003. Shapley values also credit what a model can do alone, so they divide the hybrid's 0.331 NDCG into 0.093 for EASE and about 0.06 for each of the others. Only EASE still adds a clear gain once the others are present.

NGCF's variance-inflation factor is 12.9 because it duplicates BPR. The unselected ridge fit gives BPR a negative weight for that reason. A coefficient describes the loss. The top-10 contribution is the leave-one-out change.


## Task 2.4 — Where the hybrid actually changes the list

Coefficients say how the score is written. They do not say whether the hybrid recommends different items, or whether a changed slot is a better slot. This section stays on the logistic hybrid selected in Task 2.2 and asks three mechanical questions:

1. How much of EASE's top-10 does the hybrid keep (Jaccard@10)?
2. When a slot changes, which base model ranks the new item highest, and how often is that item relevant compared with the EASE item it replaces?
3. Of the hybrid's relevant hits, how many were already an EASE hit, and how many were in nobody's top-10?

The rank of every held-out test item under EASE and under the hybrid is compared as well. The median of that change is the typical effect; the mean is reported beside it because a few items move by hundreds of ranks and pull the mean. User and item *groups* are left to Task 2.5. This section is about the mechanism of the fusion, not about who it works for.


In [19]:
# =============================================================================
# 2.4  What the selected hybrid changes, relative to EASE and to the other base models
# =============================================================================
def rank_matrix(scores):
    order = np.argsort(-scores, axis=1, kind="stable")
    rank = np.empty(order.shape, dtype=np.int32)
    rank[np.arange(order.shape[0])[:, None], order] = np.arange(order.shape[1])
    return rank


def lists_of(run_id):
    scores = np.where(CAND_TEST, np.nan_to_num(NORM_TEST[run_id], nan=-np.inf), -np.inf)
    return topk_from_scores(scores, USERS_TEST), rank_matrix(scores)


def jaccard_at_k(a, b):
    vals = []
    for user in a:
        sa, sb = set(a[user]["items"]), set(b[user]["items"])
        vals.append(len(sa & sb) / len(sa | sb))
    return float(np.mean(vals))


base_lists, base_ranks = {}, {}
for rid in HYBRID_RUN_IDS:
    base_lists[rid], base_ranks[rid] = lists_of(rid)
hyb_lists = FUSION_LISTS_TEST[PRIMARY]
hyb_ranks = rank_matrix(fused_test)
ease_lists = base_lists["tuned-EASE"]

overlap = {name: jaccard_at_k(hyb_lists, base_lists[rid])
           for name, rid in zip(SHORT, HYBRID_RUN_IDS)}
overlap["SLIMElastic"] = jaccard_at_k(hyb_lists, SELECTED["tuned-SLIMElastic"])
print("mean Jaccard@10 of the logistic hybrid with:")
for name, value in overlap.items():
    print(f"  {name:16s} {value:.3f}")

# slots where the hybrid and EASE disagree
source_hits = {m: [0, 0] for m in SHORT if m != "EASE"}  # [relevant inserts, inserts]
n_swaps, inserted_hit, removed_hit = [], [], []
for i, user in enumerate(USERS_TEST):
    rel = GT_TEST.get(user, set())
    ins = set(hyb_lists[user]["items"]) - set(ease_lists[user]["items"])
    rem = set(ease_lists[user]["items"]) - set(hyb_lists[user]["items"])
    n_swaps.append(len(ins))
    if ins:
        inserted_hit.append(np.mean([it in rel for it in ins]))
    if rem:
        removed_hit.append(np.mean([it in rel for it in rem]))
    for it in ins:
        col = ITEM_COLUMN[it]
        proposer = min((m for m in SHORT if m != "EASE"),
                       key=lambda m: base_ranks[f"tuned-{m}"][i, col])
        source_hits[proposer][1] += 1
        source_hits[proposer][0] += int(it in rel)

SWAPS = pd.DataFrame([
    {"proposer": m, "inserts": cnt, "relevant_inserts": hits,
     "precision": hits / cnt if cnt else np.nan}
    for m, (hits, cnt) in source_hits.items()
]).sort_values("inserts", ascending=False)
SWAPS.to_csv(RUNS / "tables" / "table_2_4_swaps.csv", index=False)
print(f"\nmean slots differing from EASE: {np.mean(n_swaps):.2f} of {K}")
print(f"precision of inserted items: {np.mean(inserted_hit):.3f}")
print(f"precision of the EASE items they replace: {np.mean(removed_hit):.3f}")
print(SWAPS.round(3).to_string(index=False))

# movement of every held-out relevant item, and the top-10 hit accounting
deltas, in_ease, in_hyb = [], [], []
for i, user in enumerate(USERS_TEST):
    for it in GT_TEST[user]:
        col = ITEM_COLUMN.get(it)
        if col is None or not CAND_TEST[i, col]:
            continue
        deltas.append(int(base_ranks["tuned-EASE"][i, col] - hyb_ranks[i, col]))
        in_ease.append(base_ranks["tuned-EASE"][i, col] < K)
        in_hyb.append(hyb_ranks[i, col] < K)
deltas = np.asarray(deltas)
in_ease, in_hyb = np.asarray(in_ease), np.asarray(in_hyb)
print(f"\nheld-out relevant items: {len(deltas)}")
print("rank improvement (EASE rank − hybrid rank); positive means the hybrid ranks it higher")
print("  percentiles 5/25/50/75/95:", np.percentile(deltas, [5, 25, 50, 75, 95]).round(1).tolist())
print(f"  mean {deltas.mean():.2f} | higher {(deltas > 0).mean():.3f} | "
      f"tie {(deltas == 0).mean():.3f} | lower {(deltas < 0).mean():.3f}")
print(f"entered the top-10: {int((in_hyb & ~in_ease).sum())} | "
      f"left: {int((in_ease & ~in_hyb).sum())} | "
      f"stayed: {int((in_ease & in_hyb).sum())}")


def relevant_hits(lists):
    hits = set()
    for user, rec in lists.items():
        rel = GT_TEST[user]
        for it in rec["items"]:
            if it in rel:
                hits.add((user, it))
    return hits


hit_sets = {rid: relevant_hits(lists) for rid, lists in base_lists.items()}
hyb_hits = relevant_hits(hyb_lists)
ease_hits = hit_sets["tuned-EASE"]
print(f"\nhybrid relevant top-10 hits: {len(hyb_hits)}")
print(f"  already an EASE hit: {len(hyb_hits & ease_hits)}")
print(f"  not an EASE hit: {len(hyb_hits - ease_hits)}")
print(f"  EASE hits the hybrid drops: {len(ease_hits - hyb_hits)}")
missed_by_ease = hyb_hits - ease_hits
others = [hit_sets[r] for r in HYBRID_RUN_IDS if r != "tuned-EASE"]
print(f"  of those, in nobody else's top-10 either: {len(missed_by_ease - set.union(*others))}")
print("relevant top-10 hits of each base model that no other base model also retrieves:")
for rid, hits in hit_sets.items():
    rest = set.union(*[h for k, h in hit_sets.items() if k != rid])
    unique = hits - rest
    print(f"  {rid.replace('tuned-', ''):16s} {len(unique):4d} / {len(hits):4d} ({len(unique) / len(hits):.1%})")

fig, ax = plt.subplots(figsize=(7.2, 3.8))
ax.bar(SWAPS["proposer"], SWAPS["precision"], color="#08519c")
ax.axhline(float(np.mean(removed_hit)), color="#e6550d", ls="--",
           label=f"precision of removed EASE items ({np.mean(removed_hit):.3f})")
for _, r in SWAPS.iterrows():
    ax.text(r["proposer"], r["precision"] + 0.004, f"n={int(r['inserts'])}", ha="center", fontsize=8)
ax.set_ylabel("fraction of inserted items that are relevant")
ax.set_ylim(0, max(SWAPS["precision"].max() + 0.04, 0.22))
ax.set_title("Task 2.4 - who proposes the slots the hybrid takes from EASE")
ax.legend(fontsize=8)
ax.grid(alpha=0.3, axis="y")
fig.tight_layout()
fig.savefig(RUNS / "tables" / "fig_task24.png", dpi=130, bbox_inches="tight")
plt.show()
print("saved runs/tables/table_2_4_swaps.csv, fig_task24.png")


mean Jaccard@10 of the logistic hybrid with:
  EASE             0.630
  NGCF             0.563
  ItemKNN-item     0.450
  BPR              0.504
  NeuMF            0.462
  SLIMElastic      0.533

mean slots differing from EASE: 2.38 of 10
precision of inserted items: 0.141
precision of the EASE items they replace: 0.132
    proposer  inserts  relevant_inserts  precision
        NGCF      837                95      0.114
         BPR      650               104      0.160
       NeuMF      502                85      0.169
ItemKNN-item      260                36      0.138

held-out relevant items: 9596
rank improvement (EASE rank − hybrid rank); positive means the hybrid ranks it higher
  percentiles 5/25/50/75/95: [-91.0, -9.0, 1.0, 19.0, 161.0]
  mean 17.74 | higher 0.509 | tie 0.071 | lower 0.420
entered the top-10: 320 | left: 306 | stayed: 1774

hybrid relevant top-10 hits: 2094
  already an EASE hit: 1774
  not an EASE hit: 320
  EASE hits the hybrid drops: 306
  of those, in nobod

### Reading Task 2.4

The logistic hybrid neither copies EASE nor builds a new list. Mean Jaccard@10 with EASE is 0.63, so about 2.4 of 10 slots change. Inserted items are relevant 14.1% of the time and the EASE items they replace 13.2% of the time. The swap is a wash, which is why the paired recall and NDCG gaps versus EASE include zero.

Of the hybrid's 2094 relevant top-10 hits, 1774 were already EASE hits: 320 relevant items enter and 306 leave. The median held-out item moves up by one rank. A mean gain near 18 ranks is a long tail, not the typical item. NeuMF's insertions are the most often relevant (17%) and NGCF's the least (11%), and NGCF proposes the most of them. That matches leave-one-out: NGCF's weight adds swaps the test set does not reward.

Each base model has private hits, 6–11% of its relevant top-10 items. Summing min–max scores almost never promotes an item nobody already ranked in their top-10 (8 hybrid hits). The hybrid re-votes candidates the base models already surface, and the secondary models are right about as often as EASE where the list actually changes.
